In [0]:
%pip install great-expectations
dbutils.library.restartPython()

In [0]:
import great_expectations as gx
from datetime import datetime

In [0]:
df_spark = spark.table('rfb.raw.raw_municipio_pib')

count = df_spark.count()

is_data_count_valid = count >= 1000

In [0]:
context = gx.get_context(mode="ephemeral")
df_pandas = df_spark.toPandas()

data_source = context.data_sources.add_pandas('municipio_pib_ds')
data_asset = data_source.add_dataframe_asset(name='municipio_pib_asset')
batch_definition = data_asset.add_batch_definition_whole_dataframe('municipio_pib_batch_def')

suite = context.suites.add(gx.ExpectationSuite(name='raw_municipio_pib_suite'))

# Null Expectations
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='COD_MUNICIPIO'))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='NOM_MUNICIPIO'))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column='VALOR'))

validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='raw_municipio_pib_validation',
        data=batch_definition,
        suite=suite
    )
)

gx_results = validation_definition.run(batch_parameters={'dataframe': df_pandas})

In [0]:
test_results = {
    'date_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'gx_result' : gx_results,
    'data_count' : count,
    'is_data_count_valid': is_data_count_valid
}
if gx_results.success and is_data_count_valid:
    print("="*60)
    print("Raw Municipio Pib is valid.\n")
    print(f'Test Results: {test_results}')
    print("="*60)
else:
    print(f'Test Results: {test_results}')
    raise Exception(f'ERROR: Raw Municipio Pib is invalid')